## Demo - Overall Flow

This notebook shows the overall workflow with the recommended configurations from a user's query.

* Embedding model: Titan V2 - 1024
* OpenSearch instance: c7g.large
* Reranking LLM: Nova Lite
* Parameter extraction: Nova Pro

In [1]:
import requests
import time
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.parameter_extraction import extract_parameters
from src.intent_classification import make_query_json
from src.bedrock import get_bearer_token, get_titan_response, get_cohere_response

get_bearer_token()

Bearer token set as BEARER_TOKEN_STR global variable


In [2]:
usr_query = "Who is under my medical?"

### Intent Classification

In [ ]:
# Intent classification
OPENSEARCH_DOMAIN_FQDN = 'https://vpc-int-use1-opensearch-ml-c32nkeiaudrckcr2ep7jghefje.us-east-1.es.amazonaws.com'
HEADERS = {
    'Content-Type': 'application/json',
    'Accept-Encoding': 'gzip',
}
TOP_K=5

# Titan V2 
embed_model_id = "amazon.titan-embed-text-v2:0-pgo"
# Titan V2 index - 1024
INDEX_NAME = 'search-data-titan-embed-v2_1024'

#### Embedding

In [4]:
# Query embedding
start_embed = time.time()
embed_dict = get_titan_response(usr_query, embed_model_id)
embed_vec = embed_dict["body"]["embedding"]
embed_latency = time.time() - start_embed
print( f"Embedding latency: {embed_latency}")

Embedding latency: 0.6789162158966064


#### Search

In [5]:
# Search
start_search = time.time()
query_json = make_query_json(usr_query, 
                             embed_vec=embed_vec, 
                             top_k=TOP_K, 
                             search_type='hybrid', 
                             boost_value=0.0001)

result = requests.post(f"{OPENSEARCH_DOMAIN_FQDN}/{INDEX_NAME}/_search", headers=HEADERS, json=query_json)
result_dict = json.loads(result.text)
search_latency = time.time() - start_search
print( f"Search Latency: {search_latency}")

Search Latency: 0.049506187438964844


#### (Optional) Prefiltering

In [ ]:
result_list = result_dict['hits']['hits']
top1_score = result_list[0]['_score']
top2_score = result_list[1]['_score']

if top1_score > 0.9 and (top1_score-top2_score)>0.3:
    do_rerank = False
else:
    do_rerank = True

#### LLM Reranking

In [7]:
from src.rerank import rerank_results
from src.prompt import rerank_prompt_nova

rerank_model = 'nova_pro'
rerank_prompt = rerank_prompt_nova

rerank_start = time.time()
best_result = rerank_results(usr_query, result_list, rerank_model, rerank_prompt)
rerank_latency = time.time() - rerank_start
module_id = best_result['_source']['id']

print(f"LLM Rerank Latency: {rerank_latency}")

LLM Rerank Latency: 1.0545530319213867


#### Parameter Extraction

In [8]:
from src.parameter_extraction import extract_parameters

# SOR for parameter extraction
sor_v2_intent_path = "data/sor-v2-intents-mapping.json"
with open(sor_v2_intent_path, "r") as file:
    sor_v2_intent = json.load(file)
module_keys = list(sor_v2_intent.keys())

# Parameter extraction propmt/LLM
param_prompt_version = "V3"
param_model_id = "nova_pro"

In [9]:
sor_dict = sor_v2_intent[module_id]
    
mapping = sor_dict['mapping']
func_param_name_desc = {}
func_param_name_desc["name"] = sor_dict['name']
func_param_name_desc["parameters"] = sor_dict['parameters']

ext_param_start = time.time()
ext_response = extract_parameters(user_question=usr_query, 
                              mapping_info=mapping, 
                              function_desc=func_param_name_desc, 
                              function_inst_prompt="",
                              model_id=param_model_id, 
                              dev_prompt_version=param_prompt_version)
ext_param_latency = time.time() - ext_param_start
print(ext_response)
print(f"Parameter extraction latency: {ext_param_latency}s")

{
  "arguments": {
    "standardBenefitAreas": "%22MEDICAL%22",
    "standardBenefitAreaNames": "Medical",
    "isSpecified": true
  },
  "name": "getCoveredPeopleForAssociate"
}
Parameter extraction latency: 1.3091492652893066s


#### Output

In [10]:
out_dict ={}
out_dict['usr_query'] = usr_query
out_dict['intent_id'] = module_id
out_dict['param_mapping'] = mapping
out_dict['ext_param'] = ext_response
out_dict['latency']={}
out_dict['latency']['embedding']=embed_latency
out_dict['latency']['search']=search_latency
out_dict['latency']['rerank']=rerank_latency
out_dict['latency']['param_ext']=ext_param_latency
out_dict['latency']['total']=embed_latency+search_latency+rerank_latency+ext_param_latency

out_dict

{'usr_query': 'Who is under my medical?',
 'intent_id': '46fdbb8e-2c14-46da-b1f4-3d2df5314911',
 'param_mapping': '|   standardBenefitArea   |   standardBenefitAreaName   |\n|:-----------------------:|:---------------------------:|\n| MEDICAL | Medical |\n| DENTAL | Dental |\n| VISION | Vision |\n| ADD2 | Special purpose AD&D |',
 'ext_param': '{\n  "arguments": {\n    "standardBenefitAreas": "%22MEDICAL%22",\n    "standardBenefitAreaNames": "Medical",\n    "isSpecified": true\n  },\n  "name": "getCoveredPeopleForAssociate"\n}',
 'latency': {'embedding': 0.6789162158966064,
  'search': 0.049506187438964844,
  'rerank': 1.0545530319213867,
  'param_ext': 1.3091492652893066,
  'total': 3.0921247005462646}}